# Instructor Notebook — HPFC, Retail Hedging and Procurement P&L

**Project:** Power Markets in Practice: Building and Hedging a Retail Electricity Portfolio  
**Purpose:** Instructor reference solution.

This notebook contains a complete worked solution for the summer school project. It is intentionally more complete than the student version.  
It includes data generation, data loading, HPFC construction, portfolio analysis, hedge optimization, procurement P&L, and an optional pricing extension.

The data is synthetic but designed to be plausible for a German residential electricity retail portfolio.

## 0. Instructor Notes

The project is designed around one main business story:

> A retail electricity supplier sells fixed-price electricity contracts to residential customers.  
> It must translate forward market data into hourly expected prices, understand the customer load shape, hedge using available exchange-traded products, and later evaluate procurement costs against realized Day-Ahead prices.

Pedagogical focus:
- Students should not lose time sourcing raw data.
- Students should still make decisions about which market data to use.
- The exercise should reward interpretation, not only code execution.
- AI can help them implement calculations, but they must be able to justify market assumptions.

In [ ]:
# If running in a fresh environment, install only if needed:
# !pip install pandas numpy matplotlib scipy

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize, lsq_linear
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

plt.rcParams["figure.figsize"] = (12, 5)

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

## 1. Generate Synthetic Project Data

In the final student package, you can either:
1. provide the CSV files directly, or  
2. provide this generator script only to yourself and distribute the generated files.

The data intentionally contains **more market products than strictly needed**.  
For example, 2028 quarterly quotes are included but flagged as illiquid. This lets you ask students why they did or did not use them.

In [ ]:
# Run the generator script if the CSV files are not already present.
# In the instructor package, generate_project_data.py is included next to this notebook.

import subprocess, sys, os

required_files = [
    DATA_DIR / "futures_prices.csv",
    DATA_DIR / "shape_factors.csv",
    DATA_DIR / "slp_portfolio.csv",
    DATA_DIR / "day_ahead_prices.csv",
]

if not all(p.exists() for p in required_files):
    script = BASE_DIR / "generate_project_data.py"
    if script.exists():
        subprocess.run([sys.executable, str(script)], check=True)
    else:
        print("Generator script not found. Please create the data files manually.")
else:
    print("Data files already exist.")

## 2. Load Data

In [ ]:
futures = pd.read_csv(DATA_DIR / "futures_prices.csv")
shape = pd.read_csv(DATA_DIR / "shape_factors.csv", parse_dates=["timestamp"])
slp = pd.read_csv(DATA_DIR / "slp_portfolio.csv", parse_dates=["timestamp"])
da = pd.read_csv(DATA_DIR / "day_ahead_prices.csv", parse_dates=["timestamp"])

print("futures:", futures.shape)
print("shape:", shape.shape)
print("slp:", slp.shape)
print("day ahead:", da.shape)

display(futures.head(10))
display(slp.head())

### Instructor Check

Expected observations:
- Two delivery years: 2027 and 2028.
- Hourly data for both years.
- 2027 has annual, quarterly and monthly products.
- 2028 has annual products and indicative/illiquid quarterly products.
- No monthly products for 2028.

The intended student insight:
> More granular data is useful only if it is reliable/liquid enough to be used.

In [ ]:
futures_summary = futures.groupby(["delivery_year", "delivery_period", "product_type", "is_liquid"])["price_eur_mwh"].first().reset_index()
display(futures_summary.head(30))

print("Available periods by year:")
display(futures.groupby(["delivery_year", "delivery_period"])["is_liquid"].max().reset_index())

## 3. Portfolio Analysis — SLP Load

Students should discover:
- Residential SLPs are not flat.
- Load tends to be higher in winter and evenings.
- Shape matters because electricity prices are also time-dependent.

In [ ]:
slp = slp.copy()
slp["year"] = slp["timestamp"].dt.year
slp["month"] = slp["timestamp"].dt.month
slp["hour"] = slp["timestamp"].dt.hour
slp["weekday"] = slp["timestamp"].dt.weekday
slp["is_weekend"] = slp["weekday"] >= 5

annual_load = slp.groupby("year")["load_mwh"].sum()
monthly_load = slp.groupby(["year", "month"])["load_mwh"].sum().reset_index()
hourly_profile = slp.groupby("hour")["load_mwh"].mean().reset_index()
weekday_profile = slp.groupby("weekday")["load_mwh"].mean().reset_index()

print("Annual load in MWh:")
display(annual_load)

monthly_load.pivot(index="month", columns="year", values="load_mwh").plot(marker="o")
plt.title("Monthly Portfolio Consumption")
plt.ylabel("MWh")
plt.show()

hourly_profile.plot(x="hour", y="load_mwh", marker="o", legend=False)
plt.title("Average Load by Hour of Day")
plt.ylabel("MWh")
plt.show()

weekday_profile.plot(x="weekday", y="load_mwh", kind="bar", legend=False)
plt.title("Average Load by Weekday")
plt.ylabel("MWh")
plt.show()

### Interpretation Prompt

Good answers should mention:
- Winter consumption is higher.
- Evening demand is relevant for residential customers.
- If evening and winter periods are expensive, the portfolio has positive shape risk.
- Matching total annual MWh is insufficient.

## 4. Build a Simplified HPFC

We construct an hourly price curve by combining:
- forward/futures market levels, and
- hourly shape factors.

### Suggested hierarchy for this instructor solution

For **2027**:
- Use monthly prices if available and liquid.
- Fall back to quarterly prices if monthlies are unavailable.
- Fall back to annual prices otherwise.

For **2028**:
- Use annual products only.
- Quarterly quotes exist but are flagged as illiquid, so we exclude them from the base solution.

This is an intentionally simplified approach.

In [ ]:
def parse_period_to_months(period):
    if period == "CAL":
        return list(range(1, 13))
    if period.startswith("Q"):
        q = int(period[1])
        return {1:[1,2,3], 2:[4,5,6], 3:[7,8,9], 4:[10,11,12]}[q]
    if period.startswith("M"):
        return [int(period[1:])]
    raise ValueError(f"Unknown period: {period}")

def select_forward_price(futures, year, month, product_type="BASE", use_illiquid=False):
    quotes = futures[
        (futures["delivery_year"] == year) &
        (futures["product_type"] == product_type)
    ].copy()
    if not use_illiquid:
        quotes = quotes[quotes["is_liquid"] == True]

    # Priority: monthly > quarterly > annual
    candidates = []
    for _, row in quotes.iterrows():
        months = parse_period_to_months(row["delivery_period"])
        if month in months:
            granularity_score = {"M": 3, "Q": 2, "C": 1}[row["delivery_period"][0]]
            candidates.append((granularity_score, row["price_eur_mwh"], row["delivery_period"]))

    if not candidates:
        raise ValueError(f"No forward price for {year}-{month:02d} {product_type}")

    candidates = sorted(candidates, reverse=True)
    return candidates[0][1], candidates[0][2]

def build_hpfc(shape_df, futures, product_type="BASE", use_illiquid=False):
    df = shape_df.copy()
    df["year"] = df["timestamp"].dt.year
    df["month"] = df["timestamp"].dt.month
    df["hour"] = df["timestamp"].dt.hour

    # Assign selected block forward price to each hour.
    prices = []
    periods = []
    for y, m in zip(df["year"], df["month"]):
        p, period = select_forward_price(futures, int(y), int(m), product_type, use_illiquid=use_illiquid)
        prices.append(p)
        periods.append(period)
    df["selected_forward_price"] = prices
    df["selected_forward_period"] = periods

    # Normalize shape factor within each selected block so average price equals selected forward.
    block_key = df["year"].astype(str) + "_" + df["selected_forward_period"].astype(str)
    df["block_key"] = block_key
    df["block_shape_mean"] = df.groupby("block_key")["shape_factor_annual_norm"].transform("mean")
    df[f"hpfc_{product_type.lower()}_eur_mwh"] = (
        df["selected_forward_price"] * df["shape_factor_annual_norm"] / df["block_shape_mean"]
    )
    return df

hpfc_base = build_hpfc(shape, futures, product_type="BASE", use_illiquid=False)
hpfc_peak = build_hpfc(shape, futures, product_type="PEAK", use_illiquid=False)

hpfc = hpfc_base[["timestamp", "selected_forward_price", "selected_forward_period", "hpfc_base_eur_mwh"]].copy()
hpfc = hpfc.rename(columns={
    "selected_forward_price": "selected_base_forward_price",
    "selected_forward_period": "selected_base_forward_period"
})
hpfc["hpfc_peak_eur_mwh"] = hpfc_peak["hpfc_peak_eur_mwh"]
hpfc["selected_peak_forward_price"] = hpfc_peak["selected_forward_price"]
hpfc["selected_peak_forward_period"] = hpfc_peak["selected_forward_period"]

display(hpfc.head())

### Validate HPFC Consistency

For each selected block, the average HPFC should reproduce the selected forward price.

In [ ]:
hpfc_check = hpfc.copy()
hpfc_check["year"] = hpfc_check["timestamp"].dt.year
hpfc_check["month"] = hpfc_check["timestamp"].dt.month
hpfc_check["base_block"] = hpfc_check["year"].astype(str) + "_" + hpfc_check["selected_base_forward_period"]

validation = hpfc_check.groupby("base_block").agg(
    avg_hpfc_base=("hpfc_base_eur_mwh", "mean"),
    selected_forward=("selected_base_forward_price", "first"),
    n_hours=("timestamp", "count")
)
validation["difference"] = validation["avg_hpfc_base"] - validation["selected_forward"]
display(validation.head(20))
print("Max absolute difference:", validation["difference"].abs().max())

In [ ]:
hpfc_plot = hpfc.copy()
hpfc_plot["year"] = hpfc_plot["timestamp"].dt.year
hpfc_plot["month"] = hpfc_plot["timestamp"].dt.month

monthly_hpfc = hpfc_plot.groupby(["year", "month"])["hpfc_base_eur_mwh"].mean().reset_index()
monthly_hpfc.pivot(index="month", columns="year", values="hpfc_base_eur_mwh").plot(marker="o")
plt.title("Monthly Average HPFC — Base")
plt.ylabel("EUR/MWh")
plt.show()

sample = hpfc_plot[(hpfc_plot["timestamp"] >= "2027-01-01") & (hpfc_plot["timestamp"] < "2027-01-15")]
sample.plot(x="timestamp", y="hpfc_base_eur_mwh", legend=False)
plt.title("Sample Hourly HPFC — First Two Weeks of 2027")
plt.ylabel("EUR/MWh")
plt.show()

## 5. Merge Portfolio, HPFC and Day-Ahead Data

In [ ]:
df = slp.merge(hpfc, on="timestamp", how="left").merge(da, on="timestamp", how="left")
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.weekday
df["is_peak"] = ((df["weekday"] < 5) & (df["hour"] >= 8) & (df["hour"] < 20)).astype(int)
df["is_base"] = 1

display(df.head())
print(df.isna().sum())

## 6. Expected Procurement Cost under HPFC

This gives the forward-looking cost of serving the portfolio.

In [ ]:
df["expected_cost_hpfc"] = df["load_mwh"] * df["hpfc_base_eur_mwh"]

expected_cost_by_year = df.groupby("year").agg(
    load_mwh=("load_mwh", "sum"),
    expected_cost_eur=("expected_cost_hpfc", "sum"),
)
expected_cost_by_year["expected_cost_eur_mwh"] = expected_cost_by_year["expected_cost_eur"] / expected_cost_by_year["load_mwh"]
display(expected_cost_by_year)

### Instructor Insight

The portfolio-weighted cost differs from the simple time-average HPFC because the portfolio consumes more in some hours than others.

This is a crucial teaching point:
> A flat annual price is not enough to price a shaped load profile.

In [ ]:
time_avg = df.groupby("year")["hpfc_base_eur_mwh"].mean()
load_weighted = expected_cost_by_year["expected_cost_eur_mwh"]

comparison = pd.DataFrame({
    "time_average_hpfc": time_avg,
    "load_weighted_hpfc_cost": load_weighted
})
comparison["shape_premium_eur_mwh"] = comparison["load_weighted_hpfc_cost"] - comparison["time_average_hpfc"]
display(comparison)

## 7. Hedge Optimization

We now hedge the hourly load using Base and Peak products.

### Simplified hedge profile

- Base hedge delivers 1 MWh in every hour of the delivery period.
- Peak hedge delivers 1 MWh only in peak hours.
- We optimize hedge volumes in MW-like hourly block units.

The hedge does not replicate the portfolio perfectly because the portfolio is hourly and shaped.

In [ ]:
def optimize_base_peak_for_year(data, year):
    ydf = data[data["year"] == year].copy()
    L = ydf["load_mwh"].to_numpy()
    B = np.ones(len(ydf))
    P = ydf["is_peak"].to_numpy()
    A = np.column_stack([B, P])

    if SCIPY_AVAILABLE:
        res = lsq_linear(A, L, bounds=(0, np.inf))
        q_base, q_peak = res.x
    else:
        # closed-form unconstrained, then clip
        q_base, q_peak = np.linalg.lstsq(A, L, rcond=None)[0]
        q_base, q_peak = max(q_base, 0), max(q_peak, 0)

    hedge = q_base * B + q_peak * P
    residual = L - hedge

    out = {
        "year": year,
        "q_base": q_base,
        "q_peak": q_peak,
        "portfolio_mwh": L.sum(),
        "hedge_mwh": hedge.sum(),
        "hedge_ratio": hedge.sum() / L.sum(),
        "rmse_mwh": np.sqrt(np.mean(residual**2)),
        "mae_mwh": np.mean(np.abs(residual)),
    }
    return out, hedge, residual

hedge_results = []
hedge_profiles = []

for year in sorted(df["year"].unique()):
    res, hedge, residual = optimize_base_peak_for_year(df, year)
    hedge_results.append(res)
    tmp = df[df["year"] == year].copy()
    tmp["hedge_mwh"] = hedge
    tmp["residual_mwh"] = residual
    hedge_profiles.append(tmp)

hedge_summary = pd.DataFrame(hedge_results)
hedged_df = pd.concat(hedge_profiles, ignore_index=True)

display(hedge_summary)

In [ ]:
# Visualize a sample week
sample = hedged_df[(hedged_df["timestamp"] >= "2027-01-02") & (hedged_df["timestamp"] < "2027-01-09")].copy()

plt.plot(sample["timestamp"], sample["load_mwh"], label="Portfolio load")
plt.plot(sample["timestamp"], sample["hedge_mwh"], label="Base + Peak hedge")
plt.title("Portfolio Load vs Hedge Profile — Sample Week")
plt.ylabel("MWh")
plt.legend()
plt.show()

plt.plot(sample["timestamp"], sample["residual_mwh"], label="Residual exposure")
plt.axhline(0, linewidth=1)
plt.title("Residual Exposure — Sample Week")
plt.ylabel("MWh")
plt.legend()
plt.show()

### Instructor Discussion

Expected observations:
- The hedge captures average level and some daytime/weekday structure.
- It cannot capture evening residential peaks well if those fall partly outside peak hours.
- Residual exposure remains.
- Residual is economically relevant because it is settled against volatile spot prices.

## 8. Optional: Cost-Neutral Hedge Variant

A more advanced formulation requires the hedge cost under forward prices to match the expected portfolio cost under the HPFC.

This introduces a useful discussion:
> A hedge can be chosen to match volume, cost, risk, or some combination of these — not all objectives are equivalent.

In [ ]:
def optimize_cost_neutral_base_peak(data, year):
    ydf = data[data["year"] == year].copy()
    L = ydf["load_mwh"].to_numpy()
    B = np.ones(len(ydf))
    P = ydf["is_peak"].to_numpy()
    A = np.column_stack([B, P])

    # Forward prices: use annual/month/selected price fields as simplified hourly forward levels.
    base_price = ydf["selected_base_forward_price"].to_numpy()
    peak_price = ydf["selected_peak_forward_price"].to_numpy()

    # Cost of one unit of base/peak hedge over the year.
    c_base = np.sum(B * base_price)
    c_peak = np.sum(P * peak_price)
    target_cost = np.sum(L * ydf["hpfc_base_eur_mwh"].to_numpy())

    if not SCIPY_AVAILABLE:
        return None

    def objective(q):
        residual = L - A @ q
        return np.mean(residual**2)

    cons = ({
        "type": "eq",
        "fun": lambda q: q[0] * c_base + q[1] * c_peak - target_cost
    })

    bounds = [(0, None), (0, None)]
    x0 = np.array([L.mean(), 0.0])
    res = minimize(objective, x0=x0, bounds=bounds, constraints=cons, method="SLSQP")
    return {
        "year": year,
        "success": res.success,
        "q_base": res.x[0],
        "q_peak": res.x[1],
        "message": res.message,
        "objective": res.fun,
    }

cost_neutral_results = []
if SCIPY_AVAILABLE:
    for year in sorted(df["year"].unique()):
        cost_neutral_results.append(optimize_cost_neutral_base_peak(df, year))
    display(pd.DataFrame(cost_neutral_results))
else:
    print("SciPy not available; skipping constrained optimization.")

## 9. Procurement Cost and P&L

We compare:
1. **Unhedged strategy:** buy all realized load at Day-Ahead prices.
2. **Hedged strategy:** hedge part of the load via futures, buy/sell the residual at Day-Ahead.

Simplified economic treatment:
- Hedge fixed cost is based on forward prices.
- Residual is settled at realized DA prices.
- Hedge delivery value is compared against DA as an economic P&L view.

This is not full exchange settlement/margin accounting; it is a teaching approximation.

In [ ]:
pnl = hedged_df.copy()

# Approximate hourly hedge fixed price:
# Base part valued using selected base forward price.
# Peak part valued using selected peak forward price.
# Need recover q_base/q_peak per year.
q_lookup = hedge_summary.set_index("year")[["q_base", "q_peak"]].to_dict("index")

pnl["q_base"] = pnl["year"].map(lambda y: q_lookup[y]["q_base"])
pnl["q_peak"] = pnl["year"].map(lambda y: q_lookup[y]["q_peak"])
pnl["base_hedge_mwh"] = pnl["q_base"]
pnl["peak_hedge_mwh"] = pnl["q_peak"] * pnl["is_peak"]

pnl["unhedged_cost_eur"] = pnl["load_mwh"] * pnl["day_ahead_price_eur_mwh"]

pnl["hedge_fixed_cost_eur"] = (
    pnl["base_hedge_mwh"] * pnl["selected_base_forward_price"] +
    pnl["peak_hedge_mwh"] * pnl["selected_peak_forward_price"]
)

pnl["residual_cost_eur"] = pnl["residual_mwh"] * pnl["day_ahead_price_eur_mwh"]

pnl["hedged_total_cost_eur"] = pnl["hedge_fixed_cost_eur"] + pnl["residual_cost_eur"]

# Hedge P&L view: value of hedge delivery at DA minus fixed hedge cost.
pnl["hedge_market_value_eur"] = pnl["hedge_mwh"] * pnl["day_ahead_price_eur_mwh"]
pnl["hedge_pnl_eur"] = pnl["hedge_market_value_eur"] - pnl["hedge_fixed_cost_eur"]

summary = pnl.groupby("year").agg(
    load_mwh=("load_mwh", "sum"),
    unhedged_cost_eur=("unhedged_cost_eur", "sum"),
    hedged_total_cost_eur=("hedged_total_cost_eur", "sum"),
    hedge_fixed_cost_eur=("hedge_fixed_cost_eur", "sum"),
    residual_cost_eur=("residual_cost_eur", "sum"),
    hedge_pnl_eur=("hedge_pnl_eur", "sum"),
)
summary["unhedged_eur_mwh"] = summary["unhedged_cost_eur"] / summary["load_mwh"]
summary["hedged_eur_mwh"] = summary["hedged_total_cost_eur"] / summary["load_mwh"]
summary["saving_vs_unhedged_eur"] = summary["unhedged_cost_eur"] - summary["hedged_total_cost_eur"]
summary["saving_vs_unhedged_eur_mwh"] = summary["unhedged_eur_mwh"] - summary["hedged_eur_mwh"]

display(summary)

In [ ]:
# Monthly P&L comparison
monthly = pnl.groupby(["year", "month"]).agg(
    unhedged_cost_eur=("unhedged_cost_eur", "sum"),
    hedged_total_cost_eur=("hedged_total_cost_eur", "sum"),
    hedge_pnl_eur=("hedge_pnl_eur", "sum"),
    load_mwh=("load_mwh", "sum")
).reset_index()
monthly["unhedged_eur_mwh"] = monthly["unhedged_cost_eur"] / monthly["load_mwh"]
monthly["hedged_eur_mwh"] = monthly["hedged_total_cost_eur"] / monthly["load_mwh"]

for year in sorted(monthly["year"].unique()):
    tmp = monthly[monthly["year"] == year]
    plt.plot(tmp["month"], tmp["unhedged_eur_mwh"], marker="o", label="Unhedged")
    plt.plot(tmp["month"], tmp["hedged_eur_mwh"], marker="o", label="Hedged")
    plt.title(f"Monthly Cost Comparison — {year}")
    plt.xlabel("Month")
    plt.ylabel("EUR/MWh")
    plt.legend()
    plt.show()

### Instructor Interpretation

Useful points:
- A hedge may reduce average cost in some realized scenarios and increase it in others.
- The better question is not only “did it save money?” but “did it reduce exposure to adverse price scenarios?”
- If DA prices fall below futures prices, hedging can look expensive ex post.
- If scarcity spikes occur during residual long exposure, the hedge may still leave significant risk.

## 10. Risk Metrics

Students can compute simple risk metrics:
- volatility of monthly procurement cost,
- worst month,
- best month,
- distribution of residual cost.

In [ ]:
risk = monthly.groupby("year").agg(
    unhedged_monthly_vol=("unhedged_eur_mwh", "std"),
    hedged_monthly_vol=("hedged_eur_mwh", "std"),
    worst_unhedged_month=("unhedged_eur_mwh", "max"),
    worst_hedged_month=("hedged_eur_mwh", "max"),
    best_unhedged_month=("unhedged_eur_mwh", "min"),
    best_hedged_month=("hedged_eur_mwh", "min"),
)
display(risk)

for year in sorted(pnl["year"].unique()):
    tmp = pnl[pnl["year"] == year]
    plt.hist(tmp["residual_cost_eur"], bins=80)
    plt.title(f"Distribution of Hourly Residual Cost — {year}")
    plt.xlabel("EUR")
    plt.ylabel("Frequency")
    plt.show()

## 11. Optional Extension — Retail Pricing

A simplified fixed tariff can be based on:
- expected procurement cost under HPFC,
- shape premium,
- risk premium,
- commercial margin.

We keep taxes, grid fees, levies and VAT out of scope unless explicitly added later.

In [ ]:
pricing = expected_cost_by_year.copy()
pricing["base_procurement_cost_eur_mwh"] = pricing["expected_cost_eur_mwh"]

# Example assumptions
risk_premium = 6.0      # EUR/MWh
commercial_margin = 8.0 # EUR/MWh

pricing["risk_premium_eur_mwh"] = risk_premium
pricing["commercial_margin_eur_mwh"] = commercial_margin
pricing["indicative_energy_tariff_eur_mwh"] = (
    pricing["base_procurement_cost_eur_mwh"] +
    pricing["risk_premium_eur_mwh"] +
    pricing["commercial_margin_eur_mwh"]
)
pricing["indicative_energy_tariff_ct_kwh"] = pricing["indicative_energy_tariff_eur_mwh"] / 10

display(pricing[[
    "load_mwh",
    "base_procurement_cost_eur_mwh",
    "risk_premium_eur_mwh",
    "commercial_margin_eur_mwh",
    "indicative_energy_tariff_eur_mwh",
    "indicative_energy_tariff_ct_kwh"
]])

### Instructor Discussion for Pricing

Good discussion points:
- Should the tariff use expected cost or hedged cost?
- Should different customer profiles receive different prices?
- What risk premium is appropriate if residual shape risk remains?
- How would customer churn, volume uncertainty and acquisition cost affect pricing?
- What costs are missing from this simplified tariff?

## 12. Potential Weaknesses / Things to Check Before Student Release

Use this section as a quality checklist.

### Data checks
- Are prices and loads plausible?
- Is the HPFC visually intuitive?
- Do DA prices generate interesting but not extreme P&L?
- Is the 2028 liquidity decision clear enough?

### Pedagogical checks
- Is Part 3 too hard without a helper function?
- Should students receive the period parsing logic?
- Should constrained optimization be mandatory or optional?
- Should pricing remain optional?

### Likely final design recommendation
- Student notebook should include scaffolding for loading data and plotting.
- Students should implement HPFC and hedging logic themselves.
- Instructor notebook should keep full solution and alternative approaches.